In [0]:
dbutils.library.restartPython()

In [0]:

# ===============================
# Setup Python Path for Src Folder
# ===============================
import sys
import os

# Calculate absolute path to Src folder relative to this notebook
src_path = os.path.abspath(os.path.join(os.getcwd(), "../Src"))
if src_path not in sys.path:
    sys.path.append(src_path)

print("Src folder added to Python path:", src_path)

# Import functions from Src
try:
    from DataContractValidator import load_contract, validate_table_contract
    from SilverLayer import create_silver_tables
except ModuleNotFoundError as e:
    print("Error importing modules from Src folder:", e)
    raise

# Load Contract YAML
contract_path = os.path.join(src_path, "Contracts.yaml")
contract = load_contract(contract_path)
print("Contract loaded successfully")

In [0]:
# =============================== 
# Imports & Setup 
# ===============================
from pyspark.sql.functions import *
from pyspark.sql.functions import col
from collections import Counter

# Create schemas if they don't exist
spark.sql("DROP SCHEMA IF EXISTS bronze CASCADE")
spark.sql("DROP SCHEMA IF EXISTS silver CASCADE")
spark.sql("DROP SCHEMA IF EXISTS gold CASCADE")
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")
print("Schemas created (bronze, silver, gold)")

In [0]:
# =============================== 
# Helper Functions 
# ===============================
def normalize_columns(df):
    return df.toDF(*[c.lower() for c in df.columns])

def fix_drug_schema(df):
    return (
        df
        .withColumn("cumulative_dose_to_first_reaction", col("cumulative_dose_to_first_reaction").cast("string"))
        .withColumn("drug_expiration_date", col("drug_expiration_date").cast("string"))
        .withColumn("dose_amount", col("dose_amount").cast("string"))
        .withColumn("nda_number", col("nda_number").cast("string"))
    )

def fix_null_primary_id(df):
    return df.withColumn(
        "primary_id",
        when(col("primary_id").isNull(), monotonically_increasing_id())
        .otherwise(col("primary_id"))
    )

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import desc
# -------------------------------
# Load FAERS Tables
# -------------------------------

# 2023
Demographics2023 = spark.table(
"john_snow_labs_fda_drug_adverse_events_reporting_system_faers_2023.fda_drug_adverse_events_reporting_system_faers_2023.fda_adverse_events_reporting_system_demographics_2023")

Drug2023 = spark.table(
"john_snow_labs_fda_drug_adverse_events_reporting_system_faers_2023.fda_drug_adverse_events_reporting_system_faers_2023.fda_adverse_events_reporting_system_drug_2023")

Reaction2023 = spark.table(
"john_snow_labs_fda_drug_adverse_events_reporting_system_faers_2023.fda_drug_adverse_events_reporting_system_faers_2023.fda_adverse_events_reporting_system_drug_reaction_2023")

Outcome2023 = spark.table(
"john_snow_labs_fda_drug_adverse_events_reporting_system_faers_2023.fda_drug_adverse_events_reporting_system_faers_2023.fda_adverse_events_reporting_system_patient_outcome_2023")


# 2022
Demographics2022 = spark.table(
"john_snow_labs_fda_drug_adverse_events_reporting_system_faers_2022.fda_drug_adverse_events_reporting_system_faers_2022.fda_adverse_events_reporting_system_demographics_2022")

Drug2022 = spark.table(
"john_snow_labs_fda_drug_adverse_events_reporting_system_faers_2022.fda_drug_adverse_events_reporting_system_faers_2022.fda_adverse_events_reporting_system_drug_2022")

Reaction2022 = spark.table(
"john_snow_labs_fda_drug_adverse_events_reporting_system_faers_2022.fda_drug_adverse_events_reporting_system_faers_2022.fda_adverse_events_reporting_system_drug_reaction_2022")

Outcome2022 = spark.table(
"john_snow_labs_fda_drug_adverse_events_reporting_system_faers_2022.fda_drug_adverse_events_reporting_system_faers_2022.fda_adverse_events_reporting_system_patient_outcome_2022")


# 2021
Demographics2021 = spark.table(
"john_snow_labs_fda_drug_adverse_events_reporting_system_faers_2021.fda_drug_adverse_events_reporting_system_faers_2021.fda_adverse_events_reporting_system_demographics_2021")

Drug2021 = spark.table(
"john_snow_labs_fda_drug_adverse_events_reporting_system_faers_2021.fda_drug_adverse_events_reporting_system_faers_2021.fda_adverse_events_reporting_system_drug_2021")

Reaction2021 = spark.table(
"john_snow_labs_fda_drug_adverse_events_reporting_system_faers_2021.fda_drug_adverse_events_reporting_system_faers_2021.fda_adverse_events_reporting_system_drug_reaction_2021")

Outcome2021 = spark.table(
"john_snow_labs_fda_drug_adverse_events_reporting_system_faers_2021.fda_drug_adverse_events_reporting_system_faers_2021.fda_adverse_events_reporting_system_patient_outcome_2021")


# 2020
Demographics2020 = spark.table(
"john_snow_labs_fda_drug_adverse_events_reporting_system_faers_2020.fda_drug_adverse_events_reporting_system_faers_2020.fda_adverse_events_reporting_system_demographics_2020")

Drug2020 = spark.table(
"john_snow_labs_fda_drug_adverse_events_reporting_system_faers_2020.fda_drug_adverse_events_reporting_system_faers_2020.fda_adverse_events_reporting_system_drug_2020")

Reaction2020 = spark.table(
"john_snow_labs_fda_drug_adverse_events_reporting_system_faers_2020.fda_drug_adverse_events_reporting_system_faers_2020.fda_adverse_events_reporting_system_drug_reaction_2020")

Outcome2020 = spark.table(
"john_snow_labs_fda_drug_adverse_events_reporting_system_faers_2020.fda_drug_adverse_events_reporting_system_faers_2020.fda_adverse_events_reporting_system_patient_outcome_2020")


# -------------------------------
# Normalize Column Names
# -------------------------------

Demographics2020 = normalize_columns(Demographics2020)
Demographics2021 = normalize_columns(Demographics2021)
Demographics2022 = normalize_columns(Demographics2022)
Demographics2023 = normalize_columns(Demographics2023)

Drug2020 = normalize_columns(Drug2020)
Drug2021 = normalize_columns(Drug2021)
Drug2022 = normalize_columns(Drug2022)
Drug2023 = normalize_columns(Drug2023)

Reaction2020 = normalize_columns(Reaction2020)
Reaction2021 = normalize_columns(Reaction2021)
Reaction2022 = normalize_columns(Reaction2022)
Reaction2023 = normalize_columns(Reaction2023)

Outcome2020 = normalize_columns(Outcome2020)
Outcome2021 = normalize_columns(Outcome2021)
Outcome2022 = normalize_columns(Outcome2022)
Outcome2023 = normalize_columns(Outcome2023)

print("Column names normalized")


# -------------------------------
# Fix Drug Table Schema
# -------------------------------

Drug2020_fixed = fix_drug_schema(Drug2020)
Drug2021_fixed = fix_drug_schema(Drug2021)
Drug2022_fixed = fix_drug_schema(Drug2022)
Drug2023_fixed = fix_drug_schema(Drug2023)

Drug2022_fixed = fix_null_primary_id(Drug2022_fixed)


# -------------------------------
# Combine All Years
# -------------------------------

Demographics = (
    Demographics2020
    .unionByName(Demographics2021, allowMissingColumns=True)
    .unionByName(Demographics2022, allowMissingColumns=True)
    .unionByName(Demographics2023, allowMissingColumns=True)
)

Drug = (
    Drug2020_fixed
    .unionByName(Drug2021_fixed, allowMissingColumns=True)
    .unionByName(Drug2022_fixed, allowMissingColumns=True)
    .unionByName(Drug2023_fixed, allowMissingColumns=True)
)

Reaction = (
    Reaction2020
    .unionByName(Reaction2021, allowMissingColumns=True)
    .unionByName(Reaction2022, allowMissingColumns=True)
    .unionByName(Reaction2023, allowMissingColumns=True)
)

Outcome = (
    Outcome2020
    .unionByName(Outcome2021, allowMissingColumns=True)
    .unionByName(Outcome2022, allowMissingColumns=True)
    .unionByName(Outcome2023, allowMissingColumns=True)
)


# -------------------------------
# Keep Latest Demographics Record
# -------------------------------

windowSpec = Window.partitionBy("primary_id").orderBy(desc("case_version_number"))

Demographics = (
    Demographics
    .withColumn("row_num", row_number().over(windowSpec))
    .filter(col("row_num") == 1)
    .drop("row_num")
)


# -------------------------------
# Data Validation Before Bronze
# -------------------------------

print("Drug row counts by year:")
Drug.groupBy("year").count().orderBy("year").show()

print("Demographics row counts by year:")
Demographics.groupBy("year").count().orderBy("year").show()


# -------------------------------
# Save Bronze Tables
# -------------------------------

Demographics.write.format("delta").mode("overwrite").saveAsTable("bronze.demographics")
Drug.write.format("delta").mode("overwrite").saveAsTable("bronze.drug")
Reaction.write.format("delta").mode("overwrite").saveAsTable("bronze.reaction")
Outcome.write.format("delta").mode("overwrite").saveAsTable("bronze.outcome")

print("Bronze tables created successfully")

In [0]:
# =============================== 
# Run Data Contracts 
# ===============================
# validate_table_contract(Demographics, "demographics", contract)
validate_table_contract(Drug, "drug", contract)
validate_table_contract(Reaction, "reaction", contract)
validate_table_contract(Outcome, "outcome", contract)
print("All data contracts passed")

In [0]:
import pyspark.sql.functions as F
# ===============================
# Pipeline Orchestrator
# ===============================

# Load Bronze tables
demographics = spark.table("bronze.demographics")
drug = spark.table("bronze.drug")
reaction = spark.table("bronze.reaction")
outcome = spark.table("bronze.outcome")

# Run Silver transformation
silver_tables = create_silver_tables(demographics, drug, reaction, outcome)

# Save Silver tables
silver_tables["demographics"].write.format("delta").mode("overwrite").saveAsTable("silver.demographics")
silver_tables["drugs"].write.format("delta").mode("overwrite").saveAsTable("silver.drugs")
silver_tables["reactions"].write.format("delta").mode("overwrite").saveAsTable("silver.reactions")
silver_tables["outcomes"].write.format("delta").mode("overwrite").saveAsTable("silver.outcomes")

print("Silver layer pipeline completed")

In [0]:
# =============================== 
# Gold Layer 
# ===============================

# Load Silver tables from catalog
silver_demographics = spark.table("silver.demographics")
silver_drugs = spark.table("silver.drugs")
silver_reactions = spark.table("silver.reactions")
silver_outcomes = spark.table("silver.outcomes")

# Only map years we want
year_mapping = {2020: 1, 2022: 2, 2023: 3}

# Function to map years
from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType

map_year_udf = udf(lambda y: year_mapping.get(y, None), IntegerType())

In [0]:
# ===============================
# Top 5 Drugs with the Highest 
# Adverse Event Reports Over Time
# ===============================

from pyspark.sql.functions import countDistinct, sum as spark_sum, col
from pyspark.sql.window import Window

# Load silver tables
silver_drugs = spark.table("silver.drugs")
silver_reactions = spark.table("silver.reactions")

# Compute yearly adverse event counts per drug
drug_event_counts = (
    silver_drugs
    .join(silver_reactions, ["primary_id", "case_id", "year"], "left")
    .groupBy("year", "drug_name")
    .agg(countDistinct("primary_id").alias("num_reports"))
)

# Compute total reports per drug across all years
total_reports_per_drug = (
    drug_event_counts
    .groupBy("drug_name")
    .agg(spark_sum("num_reports").alias("total_reports"))
)

# Get top 5 drugs by total reports
top5_drugs = (
    total_reports_per_drug
    .orderBy(col("total_reports").desc())
    .limit(5)
    .select("drug_name")
)

# Filter yearly table to include only top 5 drugs
drug_event_counts_top5 = (
    drug_event_counts
    .withColumn("year_id", map_year_udf(col("year")))
    .filter(col("year_id").isNotNull())  # removes 2021
    .join(top5_drugs, on="drug_name", how="inner")
    .orderBy("drug_name", "year_id")
)

# Save as gold table
drug_event_counts_top5.write.format("delta").mode("overwrite").saveAsTable("gold.drug_event_counts_top5")

# Display
display(drug_event_counts_top5)

In [0]:
# ===============================
# Most Common Drug Reactions
# ===============================

from pyspark.sql.functions import countDistinct, col
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# Load required tables
silver_drugs = spark.table("silver.drugs")
silver_reactions = spark.table("silver.reactions")

# Get the top 5 drugs from previous gold table
top5_drugs = (
    spark.table("gold.drug_event_counts_top5")
    .select("drug_name")
    .distinct()
)

# Join drugs + reactions and keep only top 5 drugs
drug_reactions = (
    silver_drugs
    .join(silver_reactions, ["primary_id", "case_id", "year"], "left")
    .join(top5_drugs, "drug_name", "inner")
)

# Count reports per drug + reaction per year
reaction_counts = (
    drug_reactions
    .groupBy("year", "drug_name", "preferred_term_for_event")
    .agg(countDistinct("primary_id").alias("num_reports"))
)

# Rank reactions per drug per year
windowSpec = Window.partitionBy("year", "drug_name").orderBy(col("num_reports").desc())

gold_top5_drug_reaction_trends = (
    reaction_counts
    .withColumn("rank", row_number().over(windowSpec))
    .filter(col("rank") <= 10)
)

# Save as gold table
gold_top5_drug_reaction_trends.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.top5_drug_reaction_trends")

display(gold_top5_drug_reaction_trends)

In [0]:
# ===============================
# Severe Outcomes by Drug
# ===============================

from pyspark.sql.functions import countDistinct, col

# Load silver tables
silver_drugs = spark.table("silver.drugs")
silver_outcomes = spark.table("silver.outcomes")

# Load the Top 5 drugs from the previous gold table
top5_drugs = (
    spark.table("gold.drug_event_counts_top5")
    .select("drug_name")
    .distinct()
)

# Severe outcome codes
severe_codes = ["DE", "HO", "LT"]

# Join drugs + outcomes and filter to Top 5 drugs
drug_outcomes = (
    silver_drugs
    .join(silver_outcomes, ["primary_id", "case_id", "year"], "left")
    .join(top5_drugs, "drug_name", "inner")
)

# Count severe outcomes
gold_top5_drug_severe_outcomes = (
    drug_outcomes
    .withColumn("year_id", map_year_udf(col("year")))
    .filter(col("year_id").isNotNull() & col("patient_outcome").isin(severe_codes))
    .groupBy("year_id", "drug_name", "patient_outcome")
    .agg(countDistinct("primary_id").alias("num_reports"))
    .orderBy("drug_name", "year_id", "patient_outcome")
)

# Save gold table
gold_top5_drug_severe_outcomes.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.top5_drug_severe_outcomes")

display(gold_top5_drug_severe_outcomes)

In [0]:
# ===============================
# Adverse Events by Age Group
# ===============================

from pyspark.sql.functions import countDistinct

# Load silver tables
silver_drugs = spark.table("silver.drugs")
silver_demographics = spark.table("silver.demographics")

# Load Top 5 drugs
top5_drugs = (
    spark.table("gold.drug_event_counts_top5")
    .select("drug_name")
    .distinct()
)

# Join drugs + demographics and filter to top 5 drugs
drug_demographics = (
    silver_drugs
    .join(silver_demographics, ["primary_id", "case_id", "year"], "left")
    .withColumn("year_id", map_year_udf(col("year")))
    .filter(col("year_id").isNotNull())
    .join(top5_drugs, "drug_name", "inner")
)

# Aggregate across all years
gold_top5_drug_age_groups = (
    drug_demographics
    .groupBy("drug_name", "age_group")
    .agg(countDistinct("primary_id").alias("num_reports"))
    .orderBy("drug_name", "num_reports")
)

# Save gold table
gold_top5_drug_age_groups.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.top5_drug_age_groups")

display(gold_top5_drug_age_groups)

In [0]:
# ===============================
# Adverse Events by Gender
# ===============================

from pyspark.sql.functions import countDistinct, col

# Load silver tables
silver_drugs = spark.table("silver.drugs")
silver_demographics = spark.table("silver.demographics")

# Load Top 5 drugs
top5_drugs = (
    spark.table("gold.drug_event_counts_top5")
    .select("drug_name")
    .distinct()
)

# Join drugs + demographics and filter to Top 5 drugs
drug_gender = (
    silver_drugs
    .join(silver_demographics, ["primary_id", "case_id", "year"], "left")
    .join(top5_drugs, "drug_name", "inner")
)

# Count reports by gender
gold_top5_drug_gender = (
    drug_gender
    .filter(col("gender_of_patient").isin("M", "F"))
    .groupBy("drug_name", "gender_of_patient")
    .agg(countDistinct("primary_id").alias("num_reports"))
    .orderBy("drug_name", "gender_of_patient")
)

# Save gold table
gold_top5_drug_gender.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.top5_drug_gender")

display(gold_top5_drug_gender)

In [0]:
# -------------------------------
# Row counts for each raw FAERS table (before bronze)
# -------------------------------

tables = {
    "Demographics2020": Demographics2020,
    "Demographics2021": Demographics2021,
    "Demographics2022": Demographics2022,
    "Demographics2023": Demographics2023,
    "Drug2020": Drug2020,
    "Drug2021": Drug2021,
    "Drug2022": Drug2022,
    "Drug2023": Drug2023,
    "Reaction2020": Reaction2020,
    "Reaction2021": Reaction2021,
    "Reaction2022": Reaction2022,
    "Reaction2023": Reaction2023,
    "Outcome2020": Outcome2020,
    "Outcome2021": Outcome2021,
    "Outcome2022": Outcome2022,
    "Outcome2023": Outcome2023
}

for name, df in tables.items():
    print(f"{name}: {df.count()} rows")